In [13]:
import pandas as pd
import os

# 1. Load the dataset
file_path = 'C:/Users/keert/OneDrive/Desktop/projects/healthcare_claim_denial/dirty_claim_data.csv' 

# Loading the data
df = pd.read_csv(file_path)

# 2. Initial Data Inspection (The "Professional" way)
print("--- Dataset Head ---")
display(df.head()) 

print("\n--- Dataset Info ---")
df.info()
df.describe()
df.shape
print("\n--- Missing Values ---")
print(df.isnull().sum())

--- Dataset Head ---


,Claim ID,Provider ID,Patient ID,Date of Service,Billed Amount,Procedure Code,Diagnosis Code,Allowed Amount,Paid Amount,Insurance Type,Claim Status,Reason Code,Follow-up Required,AR Status,Outcome
0,0HO1FSN4AP,126528997,7936697103,08/07/2024,304,99231,A02.1,218,203.0,Self-Pay,Paid,Incorrect billing information,Yes,Pending,Partially Paid
1,9U86CI2P5A,6986719948,1547160031,06/21/2024,348,99213,A16.5,216,206.0,Medicare,Paid,PRE-EXISTING CONDITION,Yes,Open,Denied
2,1QEU1AIDAU,1355108115,2611585318,07/04/2024,235,99213,A00.1,148,119.0,Commercial,Under Review,Duplicate claim,No,Denied,Denied
3,WH7XDS8CEO,9991055906,7167948632,05/26/2024,112,99215,A18.6,79,69.0,Medicare,Denied,Authorization not obtained,No,Partially Paid,Denied
4,M6OJEZ8KGI,7382167012,2140226267,07/16/2024,406,99238,A17.9,320,259.0,Medicare,Denied,Duplicate claim,No,On Hold,Denied



--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Claim ID            1000 non-null   object 
 1   Provider ID         1000 non-null   int64  
 2   Patient ID          1000 non-null   int64  
 3   Date of Service     1000 non-null   object 
 4   Billed Amount       1000 non-null   int64  
 5   Procedure Code      1000 non-null   int64  
 6   Diagnosis Code      1000 non-null   object 
 7   Allowed Amount      1000 non-null   int64  
 8   Paid Amount         950 non-null    float64
 9   Insurance Type      1000 non-null   object 
 10  Claim Status        1000 non-null   object 
 11  Reason Code         1000 non-null   object 
 12  Follow-up Required  1000 non-null   object 
 13  AR Status           1000 non-null   object 
 14  Outcome             1000 non-null   object 
dtypes: float64(1), int64(5), object(9)

In [39]:
# 3. Text Standardization (Fixing the mess)
# Strips whitespace, converts to title case, and fixes typos
df['Reason Code'] = df['Reason Code'].str.strip().str.title()
df['Reason Code'] = df['Reason Code'].replace({'Dupicate Claim': 'Duplicate Claim'})
df['Insurance Type'] = df['Insurance Type'].str.strip().str.title()

In [21]:
# 4. Handle Missing Data
# Instead of deleting, we fill with the median to maintain financial integrity
median_paid = df['Paid Amount'].median()
df['Paid Amount'] = df['Paid Amount'].fillna(median_paid)

In [25]:
# 5. Handle Outliers (Billed Amount > 5000 is likely an entry error here)
# We replace outliers with the median Billed Amount for that specific Procedure Code
df['Billed Amount'] = df['Billed Amount'].astype(float)
df.loc[df['Billed Amount'] > 5000, 'Billed Amount'] = df.groupby('Procedure Code')['Billed Amount'].transform('median')

In [43]:

## creating 3 new columns
# 1. Revenue_At_Risk
# Step A: Initialize everything to 0
df['Revenue_At_Risk'] = 0 

# Step B: Find rows where Status is 'Denied' and update ONLY those rows
df['Billed Amount'] = df['Billed Amount'].astype(float)
df['Revenue_At_Risk'] = df['Revenue_At_Risk'].astype(float)
df.loc[df['Claim Status'] == 'Denied', 'Revenue_At_Risk'] = df['Billed Amount']

# 2. Total_Potential_Revenue
df['Total_Potential_Revenue'] = df['Billed Amount']

# 3. Revenue_Gap
# Step A: Initialize everything to 0
df['Revenue_Gap'] = 0
df['Revenue_Gap'] = df['Revenue_Gap'].astype(float)

# Step B: Find rows where Status is 'Paid' and calculate the difference
# We use subtraction on the whole column at once
paid_claims = df['Claim Status'] == 'Paid'
df.loc[paid_claims, 'Revenue_Gap'] = df['Billed Amount'] - df['Paid Amount']

# Convert these specific columns to float explicitly
cols_to_convert = ['Billed Amount', 'Paid Amount', 'Revenue_At_Risk', 'Revenue_Gap']
for col in cols_to_convert:
    df[col] = df[col].astype(float)

# Now save it
df.to_csv('final_financial_claims.csv', index=False)

In [37]:
# 1. Load your final file
df_check = pd.read_csv('final_financial_claims.csv')

# 2. Check for missing values (We expect 0 now because we filled them)
print("--- Missing Values Check ---")
print(df_check.isnull().sum())

# 3. Logical Validation (The "CFO Test")
# Every 'Denied' claim MUST have a value in 'Revenue_At_Risk'
# Every 'Paid' claim MUST have a value in 'Revenue_Gap'
denied_check = df_check[df_check['Claim Status'] == 'Denied']['Revenue_At_Risk'].sum()
print(f"\n--- Financial Validation ---")
print(f"Total Revenue At Risk (Denied Claims): ${denied_check:,.2f}")

# 4. Final Preview
print("\n--- First 5 Rows of Final Data ---")
print(df_check[['Claim Status', 'Billed Amount', 'Paid Amount', 'Revenue_At_Risk', 'Revenue_Gap']].head())

--- Missing Values Check ---
Claim ID                   0
Provider ID                0
Patient ID                 0
Date of Service            0
Billed Amount              0
Procedure Code             0
Diagnosis Code             0
Allowed Amount             0
Paid Amount                0
Insurance Type             0
Claim Status               0
Reason Code                0
Follow-up Required         0
AR Status                  0
Outcome                    0
Revenue_At_Risk            0
Total_Potential_Revenue    0
Revenue_Gap                0
dtype: int64

--- Financial Validation ---
Total Revenue At Risk (Denied Claims): $97,766.50

--- First 5 Rows of Final Data ---
   Claim Status  Billed Amount  Paid Amount  Revenue_At_Risk  Revenue_Gap
0          Paid          304.0        203.0              0.0        101.0
1          Paid          348.0        206.0              0.0        142.0
2  Under Review          235.0        119.0              0.0          0.0
3        Denied         